### Project 6: Image Classification with Transfer Learning

#### Import libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt



#### Setup dataset paths (Example: Cats vs Dogs dataset from Kaggle)

In [ ]:

train_dir = "dataset/train"
val_dir = "dataset/validation"



#### Image preprocessing & augmentation

In [ ]:

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)



#### Load pre-trained model (VGG16 without top layer)

In [ ]:

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base layers (to prevent retraining)
for layer in base_model.layers:
    layer.trainable = False

#### Add custom classification head

In [ ]:

x = Flatten()(base_model.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(train_generator.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)



#### Compile model

In [ ]:

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])




#### Train model

In [ ]:

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)



##### Evaluate model

In [ ]:

loss, acc = model.evaluate(val_generator)
print(f"Validation Accuracy: {acc*100:.2f}%")



##### Plot training history

In [ ]:

plt.plot(history.history['accuracy'], label="Train Accuracy")
plt.plot(history.history['val_accuracy'], label="Val Accuracy")
plt.legend()
plt.title("Training vs Validation Accuracy")
plt.show()